In [1]:
from frameworks.LightenDiffusion.models.LightenDiffusion import Stage1, Stage2, LightenDiffusionPipeline
from frameworks.LightenDiffusion.models.unet import DiffusionUNet
from dataset_registery.registery import DatasetManager
from frameworks.LightenDiffusion.models.decom import ImageEncoder, ImageDecoder, RetinexDecomposition
from torchsummary import summary
from frameworks.LightenDiffusion.train_lightendiffusion import Stage2Trainer
from eda.helpers.data_helpers import split_dataloader
from eda.helpers.training_helpers import get_optimizer
from frameworks.LightenDiffusion.losses import stage2_loss_wrapper
import matplotlib.pyplot as plt
import os
import torch
%load_ext autoreload
%autoreload 2

/home/grads/o/omarkhater/projects/lle-generative-priors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "SICE_paired"
dataset_id = "okhater/SICE"
source = "huggingface"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [3]:
registry = DatasetManager()
registry.initialize_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "test"],
    dataset_type="paired",
    hf_cache_dir=f"../../datasets/{data_name}",
)

train_loader = registry.get_dataloader(data_name, "train", batch_size=8, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=8, shuffle=True)
train_loader, val_loader = split_dataloader(train_loader, split_ratio=0.2)

loaders = {
    "train": train_loader, 
    "val": val_loader, 
    "test": test_loader
    }
for name, loader in loaders.items():
    total_samples = len(loader.dataset)
    print(f"Loader: {name}, Total samples: {total_samples}")
    for low_imgs, label in loader:
        print(f"Batch contains {len(low_imgs)} low image samples.")
        print("Low images batch shape:", low_imgs.shape)
        print("Label batch shape:", label.shape)
        break
    print("==="*20)

Loader: train, Total samples: 207
Batch contains 8 low image samples.
Low images batch shape: torch.Size([8, 2, 3, 256, 256])
Label batch shape: torch.Size([8, 3, 256, 256])
Loader: val, Total samples: 51
Batch contains 8 low image samples.
Low images batch shape: torch.Size([8, 2, 3, 256, 256])
Label batch shape: torch.Size([8, 3, 256, 256])
Loader: test, Total samples: 46
Batch contains 8 low image samples.
Low images batch shape: torch.Size([8, 2, 3, 256, 256])
Label batch shape: torch.Size([8, 3, 256, 256])


In [4]:
directory = "/home/grads/o/omarkhater/projects/lle-generative-priors/frameworks/LightenDiffusion/trained_models/stage1/"
file_path = os.path.join(directory, "best_stage1.pth")
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")
stage1_model_weights = torch.load(file_path, map_location=device)


stage1_model = Stage1(
    encoder=ImageEncoder(64),
    decoder=ImageDecoder(64),
    decomposer=RetinexDecomposition(),
)

stage1_model.load_state_dict(stage1_model_weights)


<All keys matched successfully>

In [5]:
diffusion_net = DiffusionUNet(
    in_channels=6,
    out_channels=3,
    ch=64,
    ch_mult=(1, 2, 3, 4),
    num_res_blocks=2,
    dropout=0.0,
    conditional=True,
    resamp_with_conv=True,
)
stage2_model = Stage2(
    diffusion_unet=diffusion_net,
)
    
model = LightenDiffusionPipeline(
    stage1=stage1_model,
    stage2=stage2_model,
    
)
stage2_optimizer = get_optimizer(stage2_model)

In [6]:
trainer2 = Stage2Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=stage2_loss_wrapper,
    optimizer=stage2_optimizer,
    device=device,
    num_epochs=5,
    val_frequency =  1,
    patience = 2,
    lambda_scc=0.001,
    betas=torch.linspace(0.0001, 0.02, steps=1000),
    num_diffusion_timesteps=1000,
    num_sampling_timesteps=50,
    gamma=0.2
)
best_stage2, metrics2 = trainer2.train()


Epoch 1/5: train=0.3099, val=0.2736


Epoch 2/5: train=0.2535, val=0.2067


Epoch 3/5: train=0.1931, val=0.1876


Epoch 4/5: train=0.1897, val=0.1790


Epoch 5/5: train=0.1866, val=0.1803


In [7]:
directory = "/home/grads/o/omarkhater/projects/lle-generative-priors/frameworks/LightenDiffusion/trained_models/stage2/"
file_path = os.path.join(directory, "best_stage2.pth")
if not os.path.exists(directory):
    os.makedirs(directory)
torch.save(best_stage2.state_dict(), file_path)